<a href="https://colab.research.google.com/github/ernestoaguaysol-unpaz/sistemas-inteligentes-2026/blob/main/02_redes_neuronales/011_xor_2_capas_ocultas_lineales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

import tensorflow as tf

import matplotlib.pyplot as plt

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y = np.array([
    0,
    1,
    1,
    0
])

In [ ]:
# Creación de un modelo usando la API Sequential
modelo = tf.keras.Sequential([
    tf.keras.layers.Dense(2, input_shape=(2,), activation="linear"),
    tf.keras.layers.Dense(3, activation="linear"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [ ]:
# Compilar el modelo: definir un optimizador, función de pérdida y métricas.
modelo.compile(
    # https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/SGD
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
modelo.summary()

In [ ]:
modelo.layers

In [ ]:
modelo.layers[0]

In [ ]:
modelo.layers[0].get_weights()

In [ ]:
# Ver con distintos batch_size cómo avanza cada epoch
modelo.fit(X, y, batch_size=1, epochs=10)

In [ ]:
modelo_fit = modelo.fit(X, y, batch_size=1, epochs=50000, verbose=0)

In [ ]:
plt.plot(modelo_fit.history['loss'])

In [ ]:
plt.plot(modelo_fit.history['accuracy'])

In [ ]:
modelo.layers[0].get_weights()

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
# Variables para los bordes de decisión

# Cantidad de puntos tanto para x1 como para x2 en la grilla
# La cantidad de puntos será de (resolucion * resolucion)
# OJO: valores muy altos puede demorar el procesamiento
resolucion = 100

# Padding de valores, agregado a valores minimos y máximos de la grilla
x1_padding = 1
x2_padding = 1

def plot_borde_decision(clasificador, X, y,resolucion = 100, grafico_titulo = "clase=f(x1, x2)", es_softmax=False):
    # 0 - Si X e y son dataframes convertir a numpy
    if 'DataFrame' in str(type(X)):
        X = X.values
    if 'DataFrame' in str(type(y)):
        y = y.values

    # 1 - Calcular valores máximos y mínimos de la grilla
    x1_min, x1_max = X[:, 0].min() - x1_padding, X[:, 0].max() + x1_padding
    x2_min, x2_max = X[:, 1].min() - x2_padding, X[:, 1].max() + x2_padding

    # 2 - Calcular todos los valores x1 y x2 de la grilla
    #grilla_coordenadas_x1, grilla_coordenadas_x2 = np.meshgrid(
    #    np.linspace(x1_min, x1_max, resolucion),
    #    np.linspace(x2_min, x2_max, resolucion)
    #)
    grilla_coordenadas_x1, grilla_coordenadas_x2 = np.meshgrid(
        np.linspace(x1_min, x1_max, resolucion),
        np.linspace(x2_min, x2_max, resolucion)
    )

    # 3 - Crear X_grilla -> toda la grilla de puntos sobre la cual realizar predicciones
    # np.c_ concatena dos arrays, flatten colapsa el array a una dimensión.
    # Para crear X_grilla primero se hace .flatten() para obtener cada columna y luego se las concatena.
    X_grilla = np.c_[grilla_coordenadas_x1.flatten(), grilla_coordenadas_x2.flatten()]

    # 4 - Para plotear el contorno se debe pasara a countourf los puntos de la grilla
    # y además la "altura" o Z de contourf y debe ser de la misma forma que las
    # coordenadas (array de pares [x, y]) por lo que debe cambiarse la forma
    # de las predicciones a la forma de la grilla (ej [ypred1, ypred2])
    grilla_predicciones = clasificador.predict(X_grilla)
    if (es_softmax):
        grilla_predicciones = np.argmax(grilla_predicciones, axis=1)
    else:
        grilla_predicciones = np.where(grilla_predicciones > 0.5, 1, 0)
    grilla_alturas = grilla_predicciones.reshape(grilla_coordenadas_x1.shape)
    plt.contourf(grilla_coordenadas_x1, grilla_coordenadas_x2, grilla_alturas, cmap="Set1")


    if es_softmax:
        y_pred = np.argmax(clasificador.predict(X), axis=1)
    else:
        y_pred = clasificador.predict(X)
        y_pred = np.where(y_pred > 0.5, 1, 0)
    exactitud = np.around(accuracy_score(y, y_pred), decimals=2)

    grafico = plt.scatter(x=X[:, 0],y=X[:, 1],c=y,s=50, edgecolor='w', cmap="Set1")

    # Creación automática de elementos de la leyenda
    # https://matplotlib.org/3.5.0/gallery/lines_bars_and_markers/scatter_with_legend.html
    plt.legend(*grafico.legend_elements())

    # Textos en los ejes y título principal
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.suptitle(f"Problemática: {grafico_titulo} - Cantidad de datos: {X.shape[0]}")

    plt.title(f"Modelo: {type(clasificador).__name__} - Exactitud: {exactitud}")

    # Mostrar todo el gráfico ensamblado
    plt.show()

plot_borde_decision(modelo, X, y)

In [ ]:
y_pred = modelo.predict(X)
y_pred_binary = np.where(y_pred > 0.5, 1, 0)
y_pred_binary

In [ ]:
modelo.predict(X)

In [ ]:
modelo.evaluate(X, y)